# 04 - Gold: order facts and dashboard metrics

Materialize three Delta tables in `olist_gold`, using the preserved Bronze data,
validated Silver records and documented metric definitions.

**Run:** import this standalone notebook, select **Serverless**, keep the project's
catalog and choose **Run all**. It creates the Gold schema and replaces the three project Gold tables.

| Table | Grain | Purpose |
| --- | --- | --- |
| `fact_orders` | One accepted Silver order, any status | Detailed input for filtering and recalculation |
| `monthly_sales` | One purchase month | Monthly delivered-order item value and delivery performance |
| `sales_by_state` | One customer state | Geographic comparison of the same eligible population |

The summary tables cover the full observed period. To combine date and state filters,
use `fact_orders` and recompute metrics for the selected population.
Customer attributes are denormalized into the order fact; separate dimensions are a future extension.


## Metric contract

| Metric | Definition |
| --- | --- |
| Eligible sales order | `delivered` order with at least one valid Silver item |
| Item value | Sum of eligible item `price`, in BRL, excluding freight |
| Freight | Sum of the same items' `freight_value`, shown separately |
| Average item value per order | Item value divided by eligible order count |
| Delivery duration | Calendar days between purchase and delivery |
| On-time delivery | Delivery date on or before the promised date |
| On-time percentage | On-time orders / eligible orders with evaluable delivery deadlines × 100 |

A delivery assessment requires valid dates and a promised day not before purchase.
Unknown assessments remain null and are excluded from the rate denominator.
A zero denominator yields null. Delivery in the afternoon of the promised day is on time.

Use purchase month. These amounts do not establish accounting revenue or financial settlement.
The difference between payment and item value plus freight is an investigation field.
Missing items or payments retain null monetary amounts rather than becoming zero-value sales.

Quarantined orders and their child records are not reintroduced in Gold. Boundary months
can be partial. Recompute global rates from their numerators and denominators;
do not take an unweighted average of monthly or state rates.


In [ ]:
import re
from pyspark.sql import functions as F

def quote_identifier(name):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', name):
        raise ValueError('Use an identifier containing letters, numbers and underscores: ' + repr(name))
    return '`' + name + '`'


def table_name(catalog, layer, entity):
    return '.'.join(quote_identifier(x) for x in [catalog, 'olist_' + layer, entity])


def config(spark, dbutils):
    default_catalog = spark.sql('SELECT current_catalog()').first()[0]
    try:
        catalog = dbutils.widgets.get('catalog').strip()
    except Exception:
        dbutils.widgets.text('catalog', default_catalog, 'Existing catalog')
        catalog = dbutils.widgets.get('catalog').strip()
    quote_identifier(catalog)
    return {'catalog': catalog, 'raw_path': '/Volumes/' + catalog + '/olist_landing/raw'}


def write_snapshot(df, name):
    # Replace project-owned snapshots. Incremental MERGE is a future extension.
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(name))

In [ ]:
def build_gold_orders(spark, orders, customers, items, payments):
    """Read Silver tables and return one row for each accepted order."""
    return spark.sql(f"""
WITH item_totals AS (
  SELECT order_id, COUNT(*) AS item_rows,
         SUM(price) AS items_value, SUM(freight_value) AS freight_value
  FROM {items}
  GROUP BY order_id
), payment_totals AS (
  SELECT order_id, COUNT(*) AS payment_rows, SUM(payment_value) AS payment_value
  FROM {payments}
  GROUP BY order_id
)
SELECT o.order_id, o.customer_id, c.customer_unique_id,
       c.customer_city, c.customer_state, o.order_status,
       o.order_purchase_timestamp,
       TO_DATE(o.order_purchase_timestamp) AS purchase_date,
       TO_DATE(DATE_TRUNC('MONTH', o.order_purchase_timestamp)) AS purchase_month,
       TO_DATE(o.order_delivered_customer_date) AS delivery_date,
       TO_DATE(o.order_estimated_delivery_date) AS estimated_delivery_date,
       COALESCE(i.item_rows, 0) AS item_rows,
       i.items_value, i.freight_value,
       COALESCE(p.payment_rows, 0) AS payment_rows, p.payment_value,
       COALESCE(i.item_rows, 0) > 0 AS has_valid_items,
       COALESCE(p.payment_rows, 0) > 0 AS has_valid_payments,
       o.order_status = 'delivered' AND COALESCE(i.item_rows, 0) > 0 AS is_sales_eligible,
       p.payment_value - i.items_value - i.freight_value AS payment_minus_items_and_freight,
       CASE WHEN o.order_status = 'delivered'
                 AND o.order_delivered_customer_date IS NOT NULL
                 AND o.order_delivered_customer_date >= o.order_purchase_timestamp
            THEN DATEDIFF(TO_DATE(o.order_delivered_customer_date), TO_DATE(o.order_purchase_timestamp))
            ELSE CAST(NULL AS INT)
       END AS delivery_days,
       CASE WHEN o.order_status = 'delivered'
                 AND o.order_delivered_customer_date IS NOT NULL
                 AND o.order_delivered_customer_date >= o.order_purchase_timestamp
                 AND TO_DATE(o.order_estimated_delivery_date) >= TO_DATE(o.order_purchase_timestamp)
            THEN TO_DATE(o.order_delivered_customer_date) <= TO_DATE(o.order_estimated_delivery_date)
            ELSE CAST(NULL AS BOOLEAN)
       END AS on_time_delivery
FROM {orders} o
LEFT JOIN {customers} c ON o.customer_id = c.customer_id
LEFT JOIN item_totals i ON o.order_id = i.order_id
LEFT JOIN payment_totals p ON o.order_id = p.order_id
""")


def build_gold_summary(spark, fact_orders, grain):
    """Aggregate the eligible delivered-order population by month or state."""
    column = {'month': 'purchase_month', 'state': 'customer_state'}.get(grain)
    if column is None:
        raise ValueError('Allowed grain: month or state.')
    return spark.sql(f"""
SELECT {column},
       COUNT(*) AS delivered_orders_with_valid_items,
       SUM(items_value) AS delivered_items_value,
       SUM(freight_value) AS delivered_freight_value,
       ROUND(AVG(items_value), 2) AS average_items_value_per_order,
       ROUND(AVG(delivery_days), 2) AS average_delivery_days,
       COUNT(on_time_delivery) AS sla_evaluable_orders,
       SUM(CASE WHEN on_time_delivery THEN 1 ELSE 0 END) AS on_time_orders,
       SUM(CASE WHEN NOT on_time_delivery THEN 1 ELSE 0 END) AS late_orders,
       ROUND(100.0 * SUM(CASE WHEN on_time_delivery THEN 1 ELSE 0 END)
             / NULLIF(COUNT(on_time_delivery), 0), 2) AS on_time_rate_pct
FROM {fact_orders}
WHERE is_sales_eligible
GROUP BY {column}
""")


def gold_overview(spark, fact_orders):
    return spark.sql(f"""
SELECT COUNT(*) AS orders_in_gold,
       SUM(CASE WHEN is_sales_eligible THEN 1 ELSE 0 END) AS eligible_delivered_orders,
       SUM(CASE WHEN is_sales_eligible THEN items_value ELSE CAST(0 AS DECIMAL(18,2)) END)
         AS delivered_items_value_brl,
       COUNT(DISTINCT CASE WHEN is_sales_eligible THEN purchase_month END) AS purchase_months
FROM {fact_orders}
""")


def validate_gold(spark, sources, fact_orders, monthly_sales, sales_by_state):
    """Reconcile row grain, exact monetary totals, summaries and SLA denominators."""
    fact = spark.table(fact_orders)
    original = spark.table(sources['orders'])
    order_count = original.count()
    if fact.count() != order_count or fact.select('order_id').distinct().count() != order_count:
        raise AssertionError('Gold must contain one row per Silver order.')
    if fact.filter(F.col('order_id').isNull() | F.col('customer_unique_id').isNull()
                   | F.col('customer_state').isNull() | F.col('purchase_month').isNull()).count():
        raise AssertionError('Missing Gold key, customer, state or month.')
    if fact.filter(F.col('delivery_days') < 0).count():
        raise AssertionError('Negative delivery duration in Gold.')
    totals = fact.agg(F.sum('items_value').alias('items_value'),
                      F.sum('freight_value').alias('freight_value'),
                      F.sum('payment_value').alias('payment_value'),
                      F.sum('item_rows').alias('item_rows'),
                      F.sum('payment_rows').alias('payment_rows')).first()
    item_totals = spark.table(sources['items']).agg(F.sum('price').alias('items_value'),
                                                   F.sum('freight_value').alias('freight_value')).first()
    payment_total = spark.table(sources['payments']).agg(F.sum('payment_value').alias('value')).first()['value']
    for column in ['items_value', 'freight_value']:
        if (totals[column] or 0) != (item_totals[column] or 0):
            raise AssertionError('Duplicated or missing Gold amount: ' + column)
    if (totals['payment_value'] or 0) != (payment_total or 0):
        raise AssertionError('Duplicated or missing Gold payment amount.')
    for entity, count_column in [('items', 'item_rows'), ('payments', 'payment_rows')]:
        if (totals[count_column] or 0) != spark.table(sources[entity]).count():
            raise AssertionError('Row counts do not reconcile: ' + entity)
    eligible = fact.filter('is_sales_eligible')
    expected = eligible.agg(F.count('*').alias('orders'), F.sum('items_value').alias('items_value'),
                            F.sum('freight_value').alias('freight_value'),
                            F.count('on_time_delivery').alias('sla_orders'),
                            F.sum(F.when(F.col('on_time_delivery'), 1).otherwise(0)).alias('on_time')).first()
    result = {'orders_in_gold': order_count, 'eligible_delivered_orders': expected['orders'],
              'delivered_items_value_brl': str(expected['items_value'] or 0)}
    for name, group_column in [(monthly_sales, 'purchase_month'), (sales_by_state, 'customer_state')]:
        df = spark.table(name)
        count = df.count()
        if df.select(group_column).distinct().count() != count or df.filter(F.col(group_column).isNull()).count():
            raise AssertionError('Invalid summary grain: ' + group_column)
        actual = df.agg(F.sum('delivered_orders_with_valid_items').alias('orders'),
                         F.sum('delivered_items_value').alias('items_value'),
                         F.sum('delivered_freight_value').alias('freight_value'),
                         F.sum('sla_evaluable_orders').alias('sla_orders'),
                         F.sum('on_time_orders').alias('on_time')).first()
        for key in ['orders', 'items_value', 'freight_value', 'sla_orders', 'on_time']:
            if (actual[key] or 0) != (expected[key] or 0):
                raise AssertionError('Summary does not reconcile: ' + group_column + '.' + key)
        invalid = ((F.col('on_time_orders') + F.col('late_orders') != F.col('sla_evaluable_orders'))
                   | (F.col('sla_evaluable_orders') > F.col('delivered_orders_with_valid_items'))
                   | (F.col('on_time_rate_pct') < 0) | (F.col('on_time_rate_pct') > 100)
                   | ((F.col('sla_evaluable_orders') == 0) & F.col('on_time_rate_pct').isNotNull())
                   | ((F.col('sla_evaluable_orders') > 0) & F.col('on_time_rate_pct').isNull()))
        if df.filter(invalid).count():
            raise AssertionError('Invalid delivery-rate denominator or percentage: ' + group_column)
        result[group_column + '_rows'] = count
    return result

## 1. Check Silver inputs and prepare table names

Use the same catalog as earlier notebooks. Check required columns and the uniqueness
of order and customer-record keys before writing.


In [ ]:
cfg = config(spark, dbutils)
sources = {entity: table_name(cfg['catalog'], 'silver', entity)
           for entity in ['orders', 'customers', 'items', 'payments']}
required_columns = {
    'orders': ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
               'order_delivered_customer_date', 'order_estimated_delivery_date'],
    'customers': ['customer_id', 'customer_unique_id', 'customer_city', 'customer_state'],
    'items': ['order_id', 'price', 'freight_value'],
    'payments': ['order_id', 'payment_value'],
}
for entity, name in sources.items():
    columns = set(spark.table(name).columns)
    missing = set(required_columns[entity]) - columns
    if missing:
        raise ValueError('Missing Silver columns for ' + entity + ': ' + ', '.join(sorted(missing)))
for entity, key in [('orders', 'order_id'), ('customers', 'customer_id')]:
    source = spark.table(sources[entity])
    if source.filter(F.col(key).isNull()).count() or source.select(key).distinct().count() != source.count():
        raise AssertionError('Invalid or duplicate Silver key: ' + entity)

spark.sql('CREATE SCHEMA IF NOT EXISTS ' + quote_identifier(cfg['catalog']) + '.`olist_gold`')
fact_name = table_name(cfg['catalog'], 'gold', 'fact_orders')
monthly_name = table_name(cfg['catalog'], 'gold', 'monthly_sales')
state_name = table_name(cfg['catalog'], 'gold', 'sales_by_state')

## 2. Build one row per order

Aggregate items and payments independently by order before joining to avoid multiplying amounts.
Left joins preserve accepted orders without components and keep coverage flags explicit.


In [ ]:
fact = build_gold_orders(spark, sources['orders'], sources['customers'], sources['items'], sources['payments'])
write_snapshot(fact, fact_name)
print('Order fact written:', fact_name)

## 3. Write monthly and state summaries

Both summaries use the same eligible order population.
Retain the delivery-rate components: `on_time_orders`, `late_orders` and `sla_evaluable_orders`.


In [ ]:
write_snapshot(build_gold_summary(spark, fact_name, 'month'), monthly_name)
write_snapshot(build_gold_summary(spark, fact_name, 'state'), state_name)
print('Monthly and state summaries written.')

## 4. Validate and review the output

Require one row per Silver order. Reconcile item, freight and payment totals with Silver;
reconcile summary amounts and counts with the eligible fact population.
Check delivery-rate denominators and percentage bounds.

Reference values on the original files:

| Gold orders | Eligible orders | Item value (BRL) | Purchase months |
| ---: | ---: | ---: | ---: |
| 99,433 | 96,470 | 13,220,248.93 | 23 |

Investigate any failed assertion before consuming the dashboard metrics.
The full-period and DF/2018 dashboard reference views were also checked manually in Databricks.


In [ ]:
gold_validation = validate_gold(spark, sources, fact_name, monthly_name, state_name)
print('Validation passed:', gold_validation)
display(gold_overview(spark, fact_name))
print('Gold complete: 3 Delta tables in ' + cfg['catalog'] + '.olist_gold.')

## 5. Inspect the summaries

Review monthly item value and the customer states with the largest amounts.
State describes the customer record associated with the order, not the seller location.
All figures reflect the records accepted by this project's quality rules.


In [ ]:
display(spark.table(monthly_name).orderBy('purchase_month'))
display(spark.table(state_name).orderBy(F.desc('delivered_items_value')))

## Dashboard integration

Import `dashboard/Olist_Analytics.lvdash.json` and select the existing SQL warehouse.
The English dashboard uses the order fact so date and state filters apply before aggregation.
See `docs/DASHBOARD.md` for metrics and reference values. This notebook does not publish dashboards.

Gold writes are full snapshots, one table at a time. Avoid reading summaries during
an incomplete run. Incremental MERGE and orchestration are future extensions.

Design decisions to explain: preserving all accepted order statuses in the detailed fact,
restricting sales summaries to delivered orders with valid items, and excluding unknown
delivery assessments from the rate denominator.

Source: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce),
CC BY-NC-SA 4.0.
